In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import csv
import time
import requests
from bs4 import BeautifulSoup
import re
import os
from urllib.parse import urlparse, parse_qs
import time


from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.options import Options

headers = {
    'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win 64; x64) AppleWebKit/537.36 (KHTML, Like Gecko) Chrome/140.0.0.0 Safari/537.36 Edg/140.0.0.0'
}

In [12]:
#setup for csv (write)
CSV_HEADERS = ['title', 'tanggal', 'waktu', 'content']

#function to save to csv
def write_data_to_csv(filename, headers, data):
    try:
        with open(filename, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(headers) #  header
            writer.writerows(data)  #  data
        print(f"\n✅ SUCCESS: Total {len(data)} articles successfully saved to '{filename}'")
    except Exception as e:
        print(f"\n❌ FATAL ERROR: Failed to write data to CSV file '{filename}'. Error: {e}")

In [23]:
#setup for csv (append)
def append_rows_to_csv(filename, headers, rows):
    """
    Append baris ke CSV. Header hanya ditulis jika file belum ada atau header belum ada.
    Force flush+fsync di dalam blok 'with' supaya tidak ada operasi setelah file ditutup.
    """
    try:
        # Normalisasi rows (kalau generator/iterator, materialize biar aman)
        rows_to_write = list(rows)
        if not rows_to_write:
            print("Tidak ada baris untuk ditulis.")
            return

        write_header = True
        if os.path.exists(filename):
            # Cek apakah header sudah ada
            try:
                with open(filename, 'r', newline='', encoding='utf-8') as rf:
                    reader = csv.reader(rf)
                    first = next(reader, None)
                    write_header = (first != headers)
            except Exception:
                # Kalau gagal baca (file kosong/korup), kita tulis header lagi saja
                write_header = True

        # Tulis append
        with open(filename, 'a', newline='', encoding='utf-8') as f:
            w = csv.writer(f)
            if write_header:
                w.writerow(headers)
            w.writerows(rows_to_write)

            # Pastikan buffer flush sebelum keluar dari 'with'
            f.flush()
            try:
                os.fsync(f.fileno())
            except Exception:
                # fsync bisa gagal di beberapa FS; aman diabaikan
                pass

        print(f"💾 APPEND: {len(rows_to_write)} baris disimpan ke '{filename}'")

    except Exception as e:
        print(f"❌ ERROR APPEND CSV '{filename}': {e}")

# OKEZONE

In [4]:
urlokezone = 'https://search.okezone.com/search?q=politik&highlight=1&sort=desc&start=0'
resokezone = requests.get(urlokezone, headers=headers)

soupokezone = BeautifulSoup(resokezone.text, 'lxml')
# print(soupokezone)

In [5]:
def checklastpage(url):
    lastpage = -1
    # Setup Chrome
    chrome_options = Options()
    chrome_options.add_argument("--headless")  # jalan background
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")


    driver = webdriver.Chrome(options=chrome_options)

    try:
        driver.get(url)
        
        wait = WebDriverWait(driver, 10)
        
        try:
            pagination_elements = wait.until(EC.presence_of_all_elements_located((By.TAG_NAME, "a")))
            
            loaddata_links = []
            for element in pagination_elements:
                href = element.get_attribute('href')
                if href and 'loaddata' in href:
                    loaddata_links.append({
                        'href': href,
                        'text': element.text.strip()
                    })
            
            print(f"Found {len(loaddata_links)} loaddata links:")
            for link in loaddata_links:
                print(f"Link: {link['href']} - Text: '{link['text']}'")
            
            last_links = [link for link in loaddata_links if link['text'].lower() == 'last']
            
            if last_links:
                last_link = last_links[0]
                href = last_link['href']
                print(f"\nFound Last link: {href}")
                
                match = re.search(r'/(\d+)/?$', href)
                if match:
                    lastpage = int(match.group(1))
                    print(f"Last page number: {lastpage}")
                else:
                    print("Could not extract page number")
            else:
                print("No Last link found")
                
        except Exception as e:
            print(f"Error waiting on pagination: {e}")
            
            page_source = driver.page_source
            soup_selenium = BeautifulSoup(page_source, 'lxml')
            
            loaddata_links = soup_selenium.find_all('a', href=lambda x: x and 'loaddata' in x)
            print(f"\nFrom page source - Found {len(loaddata_links)} loaddata links:")
            
            for link in loaddata_links[:5]:
                print(f"Link: {link.get('href')} - Text: '{link.get_text().strip()}'")

    finally:
        driver.quit()
    return lastpage

In [6]:
totalpage = checklastpage(urlokezone)

Found 0 loaddata links:
No Last link found


In [7]:
print(f"page yg bisa di scrap: {totalpage}")

page yg bisa di scrap: -1


In [ ]:
# baca link berita
def getarticle_okezone(link):
  try:
    res = requests.get(link, headers=headers)
    soup = BeautifulSoup(res.text, 'lxml')
    div_content = soup.find('div', class_='c-detail read') #ganti class sesuai nama class di artikel
    paragraphs = div_content.find_all('p') #ganti tiap tag paragraf
    content = ' '.join([p.get_text(strip=True) for p in paragraphs])
    return content
    
  except Exception as e:
    print(f"Error baca artikel di: {link} | {e}")
  return ''


In [ ]:
#SCRAPING
okezone = []
counter = 0

#MAIN FUNCTION - okezone
for page in range(1, totalpage+1):
    print(f'\n--- Scraping page: {page} ---')
    url = f'https://search.okezone.com/loaddata/article/politik/{page}'
    try:
        res = requests.get(url, headers=headers)

        soup = BeautifulSoup(res.text, 'lxml')
        articles = soup.find_all('div', class_='subgroup')

        if not articles:
             print(f"No articles found on page {page}. Stopping.")
             break

        for art in articles:
            desc = art.select_one('div', class_='desc_section')
            try:
                title = desc.select_one('a').text.strip()
                link = desc.select_one('a', class_='desc-text')['href']

                #ambil tanggal
                temp = desc.find('a', class_='time-text')
                tanggal_waktu = temp.text.strip()
                tanggal = tanggal_waktu.split(' ')[0] + ' ' + tanggal_waktu.split(' ')[1] + ' ' + tanggal_waktu.split(' ')[2] #tanggalnya
                waktu = tanggal_waktu.split(' ')[3] + ' ' + tanggal_waktu.split(' ')[4] #jamnya


                content = getarticle_okezone(link)

                # Simpan data ke list
                okezone.append([title, tanggal, waktu, content])
                
                print(f'✅ Article #{counter+1} SCRAPPED! Title: {title[:50]}...')
                print(f'    100 KARAKTER PERTAMA DI CONTENT: {content[:100]} \n Waktu: {tanggal}')
                counter += 1
                time.sleep(1)
            except Exception as e:
                print(f"Error parsing artikel on page: {page} | {e}")
                continue
    except Exception as e:
        print(f"Error scraping on page: {page} | {e}")
        continue


--- Scraping page: 1 ---
✅ Article #1 SCRAPPED! Title: Pak Bas Lapor Progres IKN ke Istana, Persiapan Ibu...
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA- Kepala Otorita Ibu Kota Nusantara (OIKN) Basuki Hadimuljono merapat ke Kementerian Sekretar 
 Waktu: 03 Oktober 2025
✅ Article #2 SCRAPPED! Title: Fenomena Selebriti di Politik, Begini Pandangan Su...
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA- Dalam episode terbaru Bisikan Gaib, Robby Purba menghadirkan tamu spesial, Ki Atmo, yang di 
 Waktu: 01 Oktober 2025
✅ Article #3 SCRAPPED! Title: SAS Institute: Program MBG Bukan Janji Politik, Ta...
    100 KARAKTER PERTAMA DI CONTENT:  JAKARTA– Badan Gizi Nasional (BGN) mencatat hingga 22 September 2025, terdapat 4.711 kasus bakteri  
 Waktu: 01 Oktober 2025
✅ Article #4 SCRAPPED! Title: Husnan Bey: PPP Harus Kembali ke Khitah, Jangan Ja...
    100 KARAKTER PERTAMA DI CONTENT: JAKARTA- Muktamar X Partai Persatuan Pembangunan (PPP) yang baru saja berakhir memunculkan dualisme  
 Waktu: 30

In [ ]:
#save to csv
CSV_FILE = 'okezone_politik_articles.csv'
write_data_to_csv(CSV_FILE, CSV_HEADERS, okezone)
print(f"\n--- SCRAPING OKEZONE DONE! ({counter} article SCRAPPED!) ---")


✅ SUCCESS: Total 100 articles successfully saved to 'okezone_politik_articles.csv'

--- SCRAPING DONE! (100 article SCRAPPED!) ---


In [18]:
df = pd.read_csv('okezone_politik_articles.csv')
df.head()

,title,tanggal,waktu,content
0,"Pak Bas Lapor Progres IKN ke Istana, Persiapan...",03 Oktober 2025,16:52 WIB,JAKARTA- Kepala Otorita Ibu Kota Nusantara (OI...
1,"Fenomena Selebriti di Politik, Begini Pandanga...",01 Oktober 2025,20:55 WIB,"JAKARTA- Dalam episode terbaru Bisikan Gaib, R..."
2,SAS Institute: Program MBG Bukan Janji Politik...,01 Oktober 2025,20:20 WIB,JAKARTA– Badan Gizi Nasional (BGN) mencatat h...
3,"Husnan Bey: PPP Harus Kembali ke Khitah, Janga...",30 September 2025,18:38 WIB,JAKARTA- Muktamar X Partai Persatuan Pembangun...
4,Jokowi Sambut Baik Keputusan Prabowo Soal IKN ...,26 September 2025,16:56 WIB,SOLO– Keputusan Presiden Prabowo Subianto men...


# CNBC INDONESIA

In [15]:
# baca link berita
def get_article_cnbc(link):
    time_info_from_article = 'Time Not Found'
    content = ''
    try:
        res = requests.get(link, headers=headers)
        soup = BeautifulSoup(res.text, 'lxml')
        
        # ambil time dlu
        detail_head = soup.find('div', id='detailHead')
        if detail_head:
            # Mencari elemen waktu berdasarkan class yang spesifik
            time_tag = detail_head.find('div', class_='text-cm text-gray')
            if time_tag:
                time_info_from_article = time_tag.text.strip()
        
        #ambil konten
        div_content = soup.find('div', class_='detail-text') 
        if div_content:
            # Cari semua tag p di dalam div konten
            paragraphs = div_content.find_all('p') 
            content = ' '.join([p.get_text(strip=True) for p in paragraphs if p.get_text(strip=True)])
        
        return content, time_info_from_article
            
    except Exception as e:
        print(f"Error get article di: {link} | {e}")
    return content, time_info_from_article


In [16]:
# url list
urlcnbc = 'https://www.cnbcindonesia.com/search?query=politik'
cnbc_allpage = f'{urlcnbc}&fromdate=2015/09/01&todate=&kanal=news&tipe=artikel' 

rescnbc = requests.get(cnbc_allpage, headers=headers)
soupcnbc = BeautifulSoup(rescnbc.text, 'lxml')

In [17]:
pagenumbers = []
for a in soupcnbc.find_all('a'): #'a' bisa diganti, cek tag tag an di max page hsil search
  try:
    pagenumbers.append(int(a.get_text()))
  except:
    continue

lastpage = max(pagenumbers)
print(f"Halaman yg bisa di scraping: {lastpage}")

Halaman yg bisa di scraping: 834


In [18]:
# parsing waktuny cnbc
def parse_cnbc_time(time_info):
    time_info = time_info.replace(',', '').strip() # cleaning koma ama spasi

    # pindah ke DD Month YYYY HH:MM ( 04 October 2025 09:00)
    parts = time_info.split()
    if len(parts) >= 4:
        # Bagian tanggal ( '04 October 2025')
        tanggal = ' '.join(parts[:3]) 
        # Bagian waktu ('09:00')
        waktu = ' '.join(parts[3:]) 
        return tanggal, waktu
    
    # Fallback jika format gk dikenal
    tanggal = time_info
    waktu = 'N/A'
        
    return tanggal, waktu

In [ ]:
counter = 0
cnbc_batch = []   # buffer untuk batch tulis
# UJI: tulis tiap 10 artikel
BATCH_SIZE = 10
CSV_FILENAME = 'cnbc_politics_articles.csv'
# PRODUKSI: tulis tiap 1000 artikel
# BATCH_SIZE = 1000

for page in range(1, lastpage-831):
    url = f'{urlcnbc}&page={page}&fromdate=2015/09/01&todate=2025/09/31'
    try:
        res = requests.get(url, headers=headers, timeout=30)
        res.raise_for_status()
        soup = BeautifulSoup(res.text, 'lxml')
        articles = soup.find_all('article')

        if not articles:
            print(f"no articles found on page {page}")
            break

        for art in articles:
            try:
                a_tag = art.find('a')
                if not a_tag:
                    continue

                title = a_tag.find('h2').text.strip()
                link = a_tag['href']

                # ambil konten & waktu dari halaman detail
                content, time_info = get_article_cnbc(link)

                # parse waktu
                tanggal, waktu = parse_cnbc_time(time_info)

                # masukkan ke batch
                cnbc_batch.append([title, tanggal, waktu, content])

                print(f'✅ {counter} - {title} SCRAPPED!')
                print(f'    100 KARAKTER PERTAMA DI CONTENT: {content[:100]}\n    waktunya: {time_info}')
                counter += 1

                # ini klo dh sampe max batch, save dl ke CSV ama clear buffer
                if len(cnbc_batch) >= BATCH_SIZE:
                    append_rows_to_csv(CSV_FILENAME, CSV_HEADERS, cnbc_batch)
                    cnbc_batch.clear()

            except Exception as e:
                print(f"Error parsing artikel di hlman: {page} | {e}")
                continue

    except requests.exceptions.RequestException as e:
        print(f"Network error scraping page: {page} | {e}")
    except Exception as e:
        print(f"Error scraping (GENERAL) di hlman: {page} | {e}")
        continue

if cnbc_batch:
    append_rows_to_csv(CSV_FILENAME, CSV_HEADERS, cnbc_batch)
    cnbc_batch.clear()

✅ 0 - Mengapa Demo Gen Z Dunia Gunakan Bendera One Piece? Penjelasannya SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: Jakarta, CNBC Indonesia -Bendera bajak laut 'Topi Jerami' dari anime populer "One Piece" telah menja
    waktunya: 06 October 2025 21:00
✅ 1 - Mengenal PM Terpilih Jepang Sanae Takaichi: Murid Abe ke "Wanita Besi" SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: Jakarta, CNBC Indonesia -Panggung politik Jepang bersiap untuk menyambut era baru dengan terpilihnya
    waktunya: 06 October 2025 20:00
✅ 2 - Rupiah Perkasa Lawan Yen: Jalan-Jalan ke Jepang Bawa Rp 10 Juta Aman? SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: Jakarta, CNBC Indonesia- Nilai tukar rupiah mengalami penguatan yang cukup signifikan terhadap yen J
    waktunya: 06 October 2025 19:40
✅ 3 - Video: Di Balik Pertemuan Jokowi dan Prabowo di Kertanegara, Ada Apa? SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: Jakarta, CNBC Indonesia- Istana akhirnya menanggapi pertemuan antara Presiden Prabowo Subianto dan P

In [ ]:
print(f"\n🎉 DONNN: total artikel CNBC yang dah ke scraping: {counter}")


🎉 SELESAI: total artikel ter-scrape: 24


In [27]:
test = pd.read_csv('cnbc_politics_articles.csv')
test.head()

,title,tanggal,waktu,content
0,Mengapa Demo Gen Z Dunia Gunakan Bendera One P...,06 October 2025,21:00,"Jakarta, CNBC Indonesia -Bendera bajak laut 'T..."
1,Mengenal PM Terpilih Jepang Sanae Takaichi: Mu...,06 October 2025,20:00,"Jakarta, CNBC Indonesia -Panggung politik Jepa..."
2,Rupiah Perkasa Lawan Yen: Jalan-Jalan ke Jepan...,06 October 2025,19:40,"Jakarta, CNBC Indonesia- Nilai tukar rupiah me..."
3,Video: Di Balik Pertemuan Jokowi dan Prabowo d...,06 October 2025,19:13,"Jakarta, CNBC Indonesia- Istana akhirnya menan..."
4,"Breaking News: Baru 27 Hari Menjabat, PM Pranc...",06 October 2025,16:55,"Jakarta, CNBC Indonesia- Perdana Menteri (PM) ..."


# CNBC OLD CODE

In [7]:
counter = 0
cnbc = []

# main function to scrap
for page in range(1, lastpage+1):
  url = f'{urlcnbc}&page={page}&fromdate=2015/09/01&todate=2025/09/31' #test, dari 25 des 2024 smpe 1 jan 2025
  try:
    res = requests.get(url, headers=headers)
    soup = BeautifulSoup(res.text, 'lxml')
    articles = soup.find_all('article') #sesuaiin ke website 


    if not articles:
      print(f"no articles found on page {page}")
      break

    for art in articles:
      try:

        a_tag = art.find('a') #sesuaiin ke website (di cnbc ini ngebungkus berita, kek header gt)
        if not a_tag:
          continue

        title = a_tag.find('h2').text.strip() #sesuaiin ke website (di cnbc ini tag tag an buat judul)
        link = a_tag['href'] #sesuaiin ke website (di cnbc ini buat tembak site beritanya)
        #ambil konten ama tanggal
        content,time_info = get_article_cnbc(link)
        
        #ubah format tanggal
        tanggal, waktu = parse_cnbc_time(time_info)


        # add to cnbc
        cnbc.append([title, tanggal, waktu, content])
        
        print(f'✅ {counter} - {title} SCRAPPED!')
        print(f'    100 KARAKTER PERTAMA DI CONTENT: {content[:100]}\n waktunya: {time_info}')
        counter += 1
        # temp += 1
        # time.sleep(1)
        # if temp == 0 :
        #   time.sleep(59)
        #   temp = -100
        # if counter % 100 == 0: #setiap 100 article simpen
      
      except Exception as e:
        print(f"Error parsing artikel di hlman: {page} | {e}")
        continue
  except requests.exceptions.RequestException as e:
    print(f"Network error scraping page: {page} | {e}")
  except Exception as e:
    print(f"Error scraping (GENERAL) di hlman: {page} | {e}")
    continue

✅ 0 - Elegi Hamas, Gaza Plan, dan "Kematian" Solusi Dua Negara SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: Catatan:Artikel ini merupakan opini pribadi penulis dan tidak mencerminkan pandangan RedaksiCNBCIndo
 waktunya: 05 October 2025 18:24
✅ 1 - Habibie Sukses Turunkan Dolar dari Rp16.000 ke Rp6.550, Apa Rahasianya SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: Jakarta, CNBC Indonesia -Belum lama ini, terjadi kekeliruan teknis pada Google yang menarik perhatia
 waktunya: 05 October 2025 18:15
✅ 2 - Sejarah Militer Dunia, Pasukan yang Membuat Dunia Berubah SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: Jakarta, CNBC Indonesia— Sejarah militer dunia menyimpan banyak catatan tentang pasukan dan pemimpin
 waktunya: 05 October 2025 14:15
✅ 3 - F-35 Hingga Rafale, Inilah Daftar Senjata Paling Laku di Dunia SCRAPPED!
    100 KARAKTER PERTAMA DI CONTENT: Jakarta, CNBC Indonesia— Industri pertahanan global masih menunjukkan pola yang sama, sebagian besar
 waktunya: 05 October 2025 12:00
✅ 4

In [8]:
#save to csv
CSV_FILE = 'cnbc_politik_articles.csv'
write_data_to_csv(CSV_FILE, CSV_HEADERS, cnbc)
print(f"\n--- SCRAPING CNBC DONE! ({counter} article SCRAPPED!) ---")


✅ SUCCESS: Total 10000 articles successfully saved to 'cnbc_politik_articles.csv'

--- SCRAPING CNBC DONE! (10000 article SCRAPPED!) ---


In [10]:
df = pd.read_csv('cnbc_politik_articles_2.csv')
df.head()

,title,tanggal,waktu,content
0,"Elegi Hamas, Gaza Plan, dan ""Kematian"" Solusi ...",05 October 2025,18:24,Catatan:Artikel ini merupakan opini pribadi pe...
1,Habibie Sukses Turunkan Dolar dari Rp16.000 ke...,05 October 2025,18:15,"Jakarta, CNBC Indonesia -Belum lama ini, terja..."
2,"Sejarah Militer Dunia, Pasukan yang Membuat Du...",05 October 2025,14:15,"Jakarta, CNBC Indonesia— Sejarah militer dunia..."
3,"F-35 Hingga Rafale, Inilah Daftar Senjata Pali...",05 October 2025,12:00,"Jakarta, CNBC Indonesia— Industri pertahanan g..."
4,30 Negara yang Memilih Hidup Tanpa Militer,05 October 2025,11:00,"Jakarta, CNBC Indonesia— Di tengah meningkatny..."
